In [100]:
import os
import torch
from embeddings.resolver import get_encdec


torch.set_default_dtype(torch.float64)

In [101]:
very_old_model = True
old_ckpt = "./train_runs/encoder/bge/run_1/checkpoints/best_original.ckpt"
new_ckpt = old_ckpt.replace("best_original.ckpt", "best.ckpt")
    
dmod_ckpt = os.path.join('./dmods/encoder/ver_1', "datamodule.pth")

In [102]:
old_model = torch.load(old_ckpt, map_location='cpu', weights_only=False)
hparams = old_model['hyper_parameters']

In [103]:
new_model_args = hparams.copy()
if very_old_model:
    new_model_args.pop('loss_weights')
    new_model_args.pop('loss_fn')
    new_model_args = {
        'args': new_model_args,
        'verbose': True,
        'run_dir': os.path.dirname(os.path.dirname(old_ckpt))
    
    }

In [104]:
dm = torch.load(dmod_ckpt, map_location='cpu')

/tmp/ipykernel_2158955/2509708022.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dm = torch.load(dmod_ckpt, map_location='cpu')


In [105]:
new_model_cls = get_encdec('BGE')

In [106]:
new_model_args['args'] = new_model_cls.extract_args_from_datamodule(dm, new_model_args['args'])


{'encoder_args': {'node_feat': 3, 'edge_feat': 3, 'node_embed_dim': 8, 'edge_embed_dim': 8, 'hidden_dim': 128, 'num_layers': 3, 'heads': 2, 'set2set_steps': 3, 'latent_dim': 128, 'dropout': 0.0}, 'decoder_args': {'template_khop': 2, 'node_embed_dim': 8, 'edge_embed_dim': 8, 'hidden_dim': 128, 'num_layers': 3, 'heads': 2, 'dropout_mp': 0.0, 'dropout_mlp': 0.0, 'final_mlp_layers': 2, 'rbf_dim': 16, 'rbf_min': 0.0, 'rbf_max': 4.0, 'rbf_gamma': 10.0}, 'lr': 0.0001, 'weight_decay': 0.0, 'normIn': False, 'scheduler': True, 'scheduler_args': {'pateience': 20, 'cooldown': 20}, 'loss_latent_weight': 0.0, 'out_labels': ['bond_dist', 'angle', 'dihedral_cos', 'dihedral_sin'], 'outname': './train_runs/bge/run_2/'}


In [107]:
new_model = get_encdec('BGE')(**new_model_args)

In [108]:
new_model.hparams.keys()

dict_keys(['args', 'verbose', 'run_dir'])

In [109]:
print(old_model['state_dict'].keys())
print(new_model.state_dict().keys())

odict_keys(['normIn', 'normSet', 'Mean', 'Range', 'gnn_enc.node_embed.mlps.0.0.weight', 'gnn_enc.node_embed.mlps.0.0.bias', 'gnn_enc.node_embed.mlps.0.2.weight', 'gnn_enc.node_embed.mlps.0.2.bias', 'gnn_enc.node_embed.mlps.1.0.weight', 'gnn_enc.node_embed.mlps.1.0.bias', 'gnn_enc.node_embed.mlps.1.2.weight', 'gnn_enc.node_embed.mlps.1.2.bias', 'gnn_enc.node_embed.mlps.2.0.weight', 'gnn_enc.node_embed.mlps.2.0.bias', 'gnn_enc.node_embed.mlps.2.2.weight', 'gnn_enc.node_embed.mlps.2.2.bias', 'gnn_enc.edge_embed.mlps.0.0.weight', 'gnn_enc.edge_embed.mlps.0.0.bias', 'gnn_enc.edge_embed.mlps.0.2.weight', 'gnn_enc.edge_embed.mlps.0.2.bias', 'gnn_enc.edge_embed.mlps.1.0.weight', 'gnn_enc.edge_embed.mlps.1.0.bias', 'gnn_enc.edge_embed.mlps.1.2.weight', 'gnn_enc.edge_embed.mlps.1.2.bias', 'gnn_enc.edge_embed.mlps.2.0.weight', 'gnn_enc.edge_embed.mlps.2.0.bias', 'gnn_enc.edge_embed.mlps.2.2.weight', 'gnn_enc.edge_embed.mlps.2.2.bias', 'gnn_enc.layers.0.q_proj.weight', 'gnn_enc.layers.0.k_proj.wei

In [110]:
if very_old_model:
    old_state_dict = old_model['state_dict']
    new_state_dict = new_model.state_dict()
    compatibility_map = {
                'gnn_enc': 'encoder_net',
                'gnn_dec': 'decoder_net',
            }
    for key in old_state_dict.keys():
        new_key = key
        for old_substr, new_substr in compatibility_map.items():
            if old_substr in new_key:
                new_key = new_key.replace(old_substr, new_substr)
        new_state_dict[new_key] = old_state_dict[key]
        print(f"Updated {key} to {new_key}")
else:
    new_state_dict = old_model['state_dict']

Updated normIn to normIn
Updated normSet to normSet
Updated Mean to Mean
Updated Range to Range
Updated gnn_enc.node_embed.mlps.0.0.weight to encoder_net.node_embed.mlps.0.0.weight
Updated gnn_enc.node_embed.mlps.0.0.bias to encoder_net.node_embed.mlps.0.0.bias
Updated gnn_enc.node_embed.mlps.0.2.weight to encoder_net.node_embed.mlps.0.2.weight
Updated gnn_enc.node_embed.mlps.0.2.bias to encoder_net.node_embed.mlps.0.2.bias
Updated gnn_enc.node_embed.mlps.1.0.weight to encoder_net.node_embed.mlps.1.0.weight
Updated gnn_enc.node_embed.mlps.1.0.bias to encoder_net.node_embed.mlps.1.0.bias
Updated gnn_enc.node_embed.mlps.1.2.weight to encoder_net.node_embed.mlps.1.2.weight
Updated gnn_enc.node_embed.mlps.1.2.bias to encoder_net.node_embed.mlps.1.2.bias
Updated gnn_enc.node_embed.mlps.2.0.weight to encoder_net.node_embed.mlps.2.0.weight
Updated gnn_enc.node_embed.mlps.2.0.bias to encoder_net.node_embed.mlps.2.0.bias
Updated gnn_enc.node_embed.mlps.2.2.weight to encoder_net.node_embed.mlps.

In [111]:

new_model.load_state_dict(new_state_dict, strict=False)

_IncompatibleKeys(missing_keys=[], unexpected_keys=['Mean', 'Range'])

In [112]:
new_model.hparams.keys()

dict_keys(['args', 'verbose', 'run_dir'])

In [113]:
from pytorch_lightning import Trainer

trainer = Trainer(max_epochs=10)
trainer.validate(new_model, datamodule=dm)
# Save the final model
trainer.save_checkpoint(new_ckpt)

/mnt/projects/sne/Gohar/12.PROJECT_Surrogate_Dynamics/surrogate-dynamics/.venv/lib/python3.12/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/projects/sne/Gohar/12.PROJECT_Surrogate_Dynamic ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Validation DataLoader 0: 100%|██████████| 4/4 [00:01<00:00,  2.80it/s]

`weights_only` was not set, defaulting to `False`.



────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       val_encdec          0.002436430505564538
    val_encdec_angle       0.0013455382583882239
  val_encdec_bond_dist     0.0006551052604746441
 val_encdec_dihedral_cos  0.00017993947214861556
 val_encdec_dihedral_sin  0.00025584751455305437
        val_loss           0.002436430505564538
         val_mae           0.016319914019446057
      val_mae_angle        0.026340870877592886
    val_mae_bond_dist      0.020274072927703446
  val_mae_dihedral_cos      0.0088556317463343
  val_mae_dihedral_sin     0.009809080526153587
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
